# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [4]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq -q

import os
from groq import Groq
from google.colab import userdata
client= Groq(api_key=userdata.get('key_cHALLENGES'))
print('Cliente de Groq inicializado correctamente')

Cliente de Groq inicializado correctamente


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [6]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'La computadora funciona muy rápido', 'La computadora se apaga constantemente', 'La computadora es rápida pero la batería dura poco'"

response_zero = client.chat.completions.create(
    model ="openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': prompt_zero_shot}]
    )
print('Zero-shot:', response_zero.choices[0].message.content)


Zero-shot: **Clasificación del sentimiento**

1. *“La computadora funciona muy rápido”* → **Positivo**  
2. *“La computadora se apaga constantemente”* → **Negativo**  
3. *“La computadora es rápida pero la batería dura poco”* → **Mixto**


In [7]:
# Prompt de clasificación en modo few-shot
prompt_few_shot =""" Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto:

Reseña: "La computadora funciona muy rápido"
Reseña: "La computadora se apaga constantemente"
Reseña: "La computadora es rápida pero la batería dura poco"

"""

response_few = client.chat.completions.create(
    model ="openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': prompt_few_shot}]
    )
print('few-shot:', response_few.choices[0].message.content)

few-shot: **Sentimiento de cada reseña**

1. Reseña: *"La computadora funciona muy rápido"* → **Positivo**  
2. Reseña: *"La computadora se apaga constantemente"* → **Negativo**  
3. Reseña: *"La computadora es rápida pero la batería dura poco"* → **Mixto**


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [8]:
# Razonamiento paso a paso (chain-of-thought)
problem = (
    "Una tienda tiene una computadora con un precio de $12,000. "
    "Durante una promoción le aplican un descuento del 20%. "
    "¿Cuál es el precio final de la computadora? "
    "Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{
        "role": "user",
        "content": problem
    }]
)

print(response_cot.choices[0].message.content)


**Paso a paso**

1. **Precio original**  
   - La computadora cuesta $12 000.

2. **Calcular el descuento**  
   - Se aplica un descuento del 20 %.  
   - Para obtener el valor del descuento:  
     \[
     \text{Descuento} = 12 000 \times \frac{20}{100} = 12 000 \times 0,20 = 2 400.
     \]

3. **Restar el descuento del precio original**  
   - Precio final = Precio original – Descuento  
     \[
     12 000 - 2 400 = 9 600.
     \]

**Respuesta final**  
El precio final de la computadora, después de aplicar el descuento del 20 %, es **$9 600**.


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [9]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido=(
    "Cual fue el resultado de la feria de ciencias de Tecnologico Nacional de México 2026"
)

response_alucinacion = client.chat.completions.create(
    model ="openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': prompt_desconocido}]
    )
print(response_alucinacion.choices[0].message.content)

Lo siento, pero no puedo ayudar con eso.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [10]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [11]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
documentos = [
    "Para restablecer la contraseña, el usuario debe seleccionar la opción "
    "'Olvidé mi contraseña' e ingresar el correo registrado.",

    "Los cursos adquiridos tienen acceso durante 12 meses a partir de la fecha "
    "de compra, salvo que se indique un periodo diferente.",

    "Los certificados se generan automáticamente cuando el estudiante completa "
    "todas las lecciones y obtiene una calificación aprobatoria."
]
embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [12]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Qué tengo que hacer para obtener mi certificado?"

fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)


Fragmento recuperado: Los certificados se generan automáticamente cuando el estudiante completa todas las lecciones y obtiene una calificación aprobatoria.


In [13]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.
Política: {fragmento}
Pregunta: {pregunta}
Respuesta muy breve y corta."""
response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)
print(response_rag.choices[0].message.content)

Debes completar todas las lecciones y obtener una calificación aprobatoria.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [14]:
# Leer API key, instalar e importar librerías
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata
client= Groq(api_key=userdata.get('key_cHALLENGES'))
print('Cliente de Groq inicializado correctamente')

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

Cliente de Groq inicializado correctamente


In [17]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Android Studio es el entorno de desarrollo integrado (IDE) oficial para crear aplicaciones Android. "
    "Permite desarrollar proyectos utilizando Kotlin y proporciona herramientas para diseñar interfaces, "
    "ejecutar aplicaciones en emuladores y dispositivos físicos, depurar código y administrar dependencias.",

    "Kotlin es un lenguaje de programación moderno utilizado ampliamente para desarrollar aplicaciones Android. "
    "Su sintaxis concisa, seguridad frente a valores nulos, funciones de extensión y compatibilidad con Java "
    "facilitan la creación de aplicaciones móviles robustas y mantenibles.",

    "Jetpack Compose es un toolkit moderno de Android para construir interfaces de usuario de manera declarativa "
    "utilizando Kotlin. Permite crear pantallas mediante composables y actualizar automáticamente la interfaz "
    "cuando cambia el estado de la aplicación.",

    "Room es una biblioteca de persistencia de Android que proporciona una capa de abstracción sobre SQLite. "
    "Permite definir entidades mediante clases Kotlin, realizar consultas mediante interfaces DAO y almacenar "
    "información localmente de forma estructurada y segura.",

    "MVVM (Model-View-ViewModel) es un patrón de arquitectura utilizado en aplicaciones Android. "
    "La View representa la interfaz de usuario, el ViewModel administra el estado y la lógica de presentación, "
    "mientras que el Model se encarga de los datos. Una aplicación puede combinar MVVM con Jetpack Compose y Room "
    "para separar responsabilidades y facilitar el mantenimiento del código."
]

# Generar los embeddings
#embeddings_documentos = modelo_embeddings.encode(documentos)

#print("Cantidad de documentos:", len(documentos))
#print("Dimensiones de los embeddings:", embeddings_documentos.shape)



embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (5, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [18]:
# Definir la función buscar_fragmento
from sklearn.metrics.pairwise import cosine_similarity

def buscar_fragmento(pregunta):
    # Generar embedding de la pregunta
    embedding_pregunta = modelo_embeddings.encode([pregunta])

    # Calcular similitud coseno entre la pregunta y los documentos
    similitudes = cosine_similarity(
        embedding_pregunta,
        embeddings_documentos
    )[0]

    # Obtener el índice del documento más similar
    indice_mas_relevante = np.argmax(similitudes)

    # Obtener el fragmento más relevante
    fragmento = documentos[indice_mas_relevante]

    # Obtener el valor de similitud
    similitud = similitudes[indice_mas_relevante]

    return fragmento, similitud


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [19]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
# Pregunta real sobre los documentos, sin proporcionar contexto
pregunta = "¿Cómo se construyen las interfaces de usuario utilizando Jetpack Compose?"

response_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": pregunta
        }
    ]
)

respuesta_sin_rag = response_sin_rag.choices[0].message.content

print("Respuesta sin RAG:")
print(respuesta_sin_rag)


Respuesta sin RAG:
## 1. ¿Qué es Jetpack Compose?

Jetpack Compose es el framework de UI declarativo de Android.  
En lugar de escribir XML, defines la pantalla como **funciones** (`@Composable`) que describen *qué* se debe mostrar. Cada vez que cambia el estado, Compose vuelve a compilar solo las partes afectadas.  

Algunas ventajas:
- **Código más compacto y legible**  
- **Preview en tiempo real** (Android Studio)  
- **Mejor integración con Kotlin** (extensiones, corutinas, `State`, `Flow`, etc.)  
- **Material 3 listo para usar**  

---

## 2. Primeros pasos: configuración

| Paso | Acción | Código / Archivo |
|------|--------|------------------|
| 1 | **Crear proyecto** | Android Studio → “New Project” → “Empty Compose Activity” |
| 2 | **Añadir dependencias** | `build.gradle (app)`<br>`implementation "androidx.compose.ui:ui:1.6.0"`<br>`implementation "androidx.compose.material3:material3:1.2.0"`<br>`implementation "androidx.compose.ui:ui-tooling-preview:1.6.0"`<br>`debugImpleme

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [20]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
# Pregunta sobre los documentos
pregunta = "¿Cómo se construyen las interfaces de usuario utilizando Jetpack Compose?"

# Recuperar el fragmento más relevante
fragmento, similitud = buscar_fragmento(pregunta)

print("Fragmento recuperado:")
print(fragmento)
print(f"\nSimilitud: {similitud:.4f}")


# Crear el prompt utilizando RAG
prompt_rag = f"""
Responde la siguiente pregunta utilizando únicamente la información
proporcionada en el fragmento recuperado.

Fragmento:
{fragmento}

Pregunta:
{pregunta}

Si el fragmento no contiene información suficiente para responder,
indícalo claramente.
"""


# Enviar pregunta + fragmento al modelo
response_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_rag
        }
    ]
)

# Guardar la respuesta
respuesta_con_rag = response_con_rag.choices[0].message.content

print("\nRespuesta con RAG:")
print(respuesta_con_rag)


Fragmento recuperado:
Jetpack Compose es un toolkit moderno de Android para construir interfaces de usuario de manera declarativa utilizando Kotlin. Permite crear pantallas mediante composables y actualizar automáticamente la interfaz cuando cambia el estado de la aplicación.

Similitud: 0.7808

Respuesta con RAG:
Se construyen de manera **declarativa** usando **composables** en Kotlin. Con Jetpack Compose, se crean pantallas mediante estos composables y la interfaz se actualiza automáticamente cuando cambia el estado de la aplicación.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [21]:
# Mostrar ambas respuestas para comparar
# Mostrar resultados de la comparación

print("========== PREGUNTA ==========")
print(pregunta)

print("\n========== FRAGMENTO RECUPERADO ==========")
print(fragmento)

print("\n========== RESPUESTA SIN RAG ==========")
print(respuesta_sin_rag)

print("\n========== RESPUESTA CON RAG ==========")
print(respuesta_con_rag)

print(
    "La respuesta con RAG es más precisa porque el modelo recibió "
    "un fragmento relevante de la base de conocimiento como contexto. "
    "Esto reduce la posibilidad de generar información incorrecta o "
    "inventada (alucinaciones). La respuesta sin RAG depende únicamente "
    "del conocimiento disponible en el modelo."
)

========== PREGUNTA ==========
¿Cómo se construyen las interfaces de usuario utilizando Jetpack Compose?

========== FRAGMENTO RECUPERADO ==========
Jetpack Compose es un toolkit moderno de Android para construir interfaces de usuario de manera declarativa utilizando Kotlin. Permite crear pantallas mediante composables y actualizar automáticamente la interfaz cuando cambia el estado de la aplicación.

========== RESPUESTA SIN RAG ==========
## 1. ¿Qué es Jetpack Compose?

Jetpack Compose es el framework de UI declarativo de Android.  
En lugar de escribir XML, defines la pantalla como **funciones** (`@Composable`) que describen *qué* se debe mostrar. Cada vez que cambia el estado, Compose vuelve a compilar solo las partes afectadas.  

Algunas ventajas:
- **Código más compacto y legible**  
- **Preview en tiempo real** (Android Studio)  
- **Mejor integración con Kotlin** (extensiones, corutinas, `State`, `Flow`, etc.)  
- **Material 3 listo para usar**  

---

## 2. Primeros pasos: co